In [ ]:
import pandas as pd
import numpy as np
import time
from datetime import datetime
import os

In [ ]:
def get_adb_means(df_adb, df_hrv, window_size):
    df_adb['recordTime'] = pd.to_datetime(df_adb['recordTime'], format='%Y%m%d%H%M%S')
    df_adb['unixTime'] = (df_adb['recordTime'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')
    df_hrv.reset_index(drop=True, inplace=True)
    ADB_Times = []
    pd.options.display.float_format = '{:.0f}'.format  # nearest whole number for unixTime
    columns_to_search = ['isAggressiveStreering', 'isHardAcc', 'isHardBrak', 'isSpeeding']
    for column in columns_to_search:
        true_values = df_adb[df_adb[column] == True]
        if not true_values.empty:
            ADB_Times += true_values['unixTime'].tolist()
    print(ADB_Times)

    # Convert Unix timestamps to datetime objects and store in a new list
    ADB_datetimes = [datetime.fromtimestamp(t) for t in ADB_Times]

    event_data = []  # List to store the data for each ADB event
    
    for adb_time in ADB_Times:
        # Calculate the start and end times for the 5-minute window before the ADB event
        end_time = pd.to_datetime(adb_time, unit='s')
        start_time = end_time - window_size

        # Extract the rows within the 5-minute window
        window_adb = df_adb[(df_adb['unixTime'] >= start_time.timestamp()) & (df_adb['unixTime'] <= end_time.timestamp())]

        last_unix_time = window_adb.iloc[-1]['unixTime']
        interval_start = last_unix_time - 300  # Calculate the start of the 5-minute interval

        interval_means = []  # List to store the mean values for each interval

        for i in range(0, 300):
            # Calculate the start time for the current 5-minute interval
            current_start = interval_start - i
            current_end = current_start + 300

            # Filter df_hrv based on the interval
            interval_data = df_hrv[(df_hrv['end_unix'] >= current_start) & (df_hrv['end_unix'] < current_end)]

            if not interval_data.empty:
                # Calculate the mean for each column in the interval_data DataFrame
                interval_mean = interval_data.mean()
                interval_means.extend(interval_mean.values)  # Add mean values to the list
            else:
                interval_means.extend([np.nan] * len(df_hrv.columns))  # Add NaN values for missing intervals
        event_data.append(interval_means)  # Append mean values for the current ADB event to the event_data list
    
    final_df = pd.DataFrame(event_data)
    
    return final_df, ADB_Times

In [ ]:
def non_adb_means(df_adb, df_hrv, removal_window, ADB_Times, window_size=pd.Timedelta(minutes=5), overlap_size=pd.Timedelta(minutes=2.5), pct_start_moving=0.8):
    df_adb['recordTime'] = pd.to_datetime(df_adb['recordTime'], format='%Y%m%d%H%M%S')
    df_adb['unixTime'] = (df_adb['recordTime'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s') 
    event_data = []  # List to store the data for each ADB event
    # If there are no ADB cases, return the original DataFrame
    if len(ADB_Times) == 0:
        df_remaining = df_adb 
    else:
        # REMOVE 30 MIN WINDOW BEFORE ADB
        adb_remove = []
        if len(ADB_Times) == 1:
            adb_time = ADB_Times[0]
            end_time = pd.to_datetime(adb_time, unit='s')
            start_time = end_time - window_size
            adb_remove = df_adb[(df_adb['unixTime'] >= start_time.timestamp()) & (df_adb['unixTime'] <= end_time.timestamp())]
        else:
            for adb_time in ADB_Times:    
                end_time = pd.to_datetime(adb_time, unit='s')
                start_time = end_time - window_size
                window_remove = df_adb[(df_adb['unixTime'] >= start_time.timestamp()) & (df_adb['unixTime'] <= end_time.timestamp())]
                adb_remove.append(window_remove)
            adb_remove = pd.concat(adb_remove).reset_index(drop=True)

        mask = df_adb.isin(adb_remove)
        df_remaining = df_adb[~mask.any(axis=1)]

    # overlapping intervals
    overlap_seconds = overlap_size.total_seconds()
    windows = []
    i = 0
    while i < len(df_remaining):
        start_time = df_remaining.iloc[i]['recordTime']
        end_time = start_time + window_size
        window = df_remaining[(df_remaining['recordTime'] >= start_time) & (df_remaining['recordTime'] < end_time)]
        windows.append(window)
        i += int(window_size.total_seconds() - overlap_seconds)

    # Filter windows based on moving time percentage
    filtered_windows = []
    for window in windows:
        pct_start_moving_window = window['isStartMoving'].mean()
        if pct_start_moving_window >= pct_start_moving:
            filtered_windows.append(window)

    for window in filtered_windows:
        last_unix_time = window.iloc[-1]['unixTime']
        interval_start = last_unix_time - 300  # Calculate the start of the 5-minute interval

        interval_means = []  # List to store the mean values for each interval

        for i in range(0, 300):
            # Calculate the start time for the current 5-minute interval
            current_start = interval_start - i
            current_end = current_start + 300

            # Filter df_hrv based on the interval
            interval_data = df_hrv[(df_hrv['end_unix'] >= current_start) & (df_hrv['end_unix'] < current_end)]

            if not interval_data.empty:
                # Calculate the mean for each column in the interval_data DataFrame
                interval_mean = interval_data.mean()
                interval_means.extend(interval_mean.values)  # Add mean values to the list
            else:
                interval_means.extend([np.nan] * len(df_hrv.columns))  # Add NaN values for missing intervals
        event_data.append(interval_means)  # Append mean values for the current ADB event to the event_data list
    
    final_df = pd.DataFrame(event_data)

    return final_df

In [ ]:
DB_file_path = "C:/Users/wzqwa/OneDrive - Imperial College London/Imperial Year 4/FYP/Data example/Provided DB_Final.xlsx"
DB_Final = pd.read_excel(DB_file_path, sheet_name="Summary-final")
DB_adb = DB_Final[:-13]
Dates = DB_adb["Driving date"].tolist()
driver_no = DB_adb["Name"].astype(int).tolist()  
ODI = DB_adb["ODI-3%"].astype(int).tolist()  
CVHRI = DB_adb["CVHRI"].astype(int).tolist()  
CEI = DB_adb["CEI"].astype(int).tolist()  

dates = [date.strftime('%Y-%m-%d') for date in Dates]
DB_df = pd.DataFrame({'NO': driver_no, 'ODI-3%': ODI, 'CVHRI': CVHRI, 'CEI':CEI, 'dates': dates})
print(DB_df)

In [ ]:
directory = r"C:\Users\wzqwa\OneDrive - Imperial College London\Imperial Year 4\FYP\Data example\ALL_TRIP_DATA"
file_names = []
dummyf = []
for root, dirs, files in os.walk(directory):
    for file in files:
        if file.endswith(".xlsx"):
            file_names.append(file)
            dummyf.append(root)
print(len(dummyf))

In [ ]:
import warnings
warnings.filterwarnings("ignore")
    
adb_mean_list = []
non_adb_mean_list = []
window_size = pd.Timedelta(minutes=5)
overlap_size = pd.Timedelta(minutes=2.5)
removal_window = pd.Timedelta(minutes=30)
pct_start_moving = 0.5

for root, filename in zip(dummyf, file_names):
    file_name_no_ext = os.path.splitext(filename)[0]
    if file_name_no_ext in DB_df['dates'].values:  # Check if the date is present in DB_df
        file_path = os.path.join(root, filename)
        df_adb = pd.read_excel(file_path)
        print(filename)  
        file_name = file_path.split("\\")[-3]
        driver_no = file_name.strip(".")

        # Construct the file path for df_hrv using the corresponding driver_no
        hrv_file_path = f'C:/Users/wzqwa/OneDrive - Imperial College London/Imperial Year 4/FYP/Data example/final_data/Final_{driver_no}.csv'
        df_hrv_raw = pd.read_csv(hrv_file_path)
        df_hrv = df_hrv_raw.iloc[:, 3:]
        adb_mean_df, ADB_Times = get_adb_means(df_adb, df_hrv, window_size)
        non_adb_mean_df = non_adb_means(df_adb, df_hrv, removal_window, ADB_Times, window_size, overlap_size, pct_start_moving)

        adb_mean_list.append(adb_mean_df)
        non_adb_mean_list.append(non_adb_mean_df)


#####################################
        driver_info = DB_df[DB_df['dates'] == file_name_no_ext]  # Get driver info from DB_df

        odi_columns = [f'ODI-3%_{i}' for i in range(0, 300)]
        cvhri_columns = [f'CVHRI_{i}' for i in range(0, 300)]
        cei_columns = [f'CEI_{i}' for i in range(0, 300)]

        driver_info = DB_df[DB_df['dates'] == file_name_no_ext]  # Get driver info from DB_df

        for odi_col in odi_columns:
            adb_mean_df[odi_col] = driver_info['ODI-3%'].values[0]  # Add 'ODI-3%' column
            non_adb_mean_df['ODI-3%'] = driver_info['ODI-3%'].values[0]  # Add 'ODI-3%' column
        for cvhri_col in cvhri_columns:
            adb_mean_df[cvhri_col] = driver_info['CVHRI'].values[0]  # Add 'CVHRI' column
            non_adb_mean_df['CVHRI'] = driver_info['CVHRI'].values[0]  # Add 'CVHRI' column
        for cei_col in cei_columns:
            adb_mean_df[cei_col] = driver_info['CEI'].values[0]  # Add 'CEI' column
            non_adb_mean_df['CEI'] = driver_info['CEI'].values[0]  # Add 'CEI' column
###################################

adb_mean = pd.concat(adb_mean_list, axis=0).reset_index(drop=True)
non_adb_mean= pd.concat(non_adb_mean_list, axis=0).reset_index(drop=True)

print(adb_mean)
print(non_adb_mean)

In [ ]:
sleep_data_adb = adb_mean.iloc[:, -900:]  
adb_data_df = adb_mean.iloc[:, :-900]
print(adb_data_df)

In [ ]:
sleep_data_non_adb = non_adb_mean.iloc[:,-3:]
non_adb_df = non_adb_mean.iloc[:,:-3]
print(non_adb_df)

In [ ]:
columns_to_copy = sleep_data_non_adb.columns
column_copies = [np.tile(sleep_data_non_adb[column].values[:, np.newaxis], (1, 300)) for column in columns_to_copy]

# Create a new data frame with repeated columns
new_columns = [f'{column}_{i}' for column in columns_to_copy for i in range(300)]
sleep_data_non_adb = pd.DataFrame(data=np.concatenate(column_copies, axis=1), columns=new_columns)

print(sleep_data_non_adb)

In [ ]:
titles = []  # Declare an empty list for titles before the loop starts
column_names = ['start_unix', 'end_unix', 'mean_nni', 'sdnn', 'sdsd', 'nni_50', 'pnni_50', 'nni_20', 'pnni_20', 'rmssd', 'median_nni', 'range_nni', 'cvsd', 'cvnni', 'mean_hr', 'max_hr', 'min_hr', 'std_hr', 'lf', 'hf', 'lf_hf_ratio', 'lfnu', 'hfnu', 'total_power', 'vlf']

for i in range(0, 300):
    new_column_names = [f"{column}-{i}" for column in column_names]
    titles.extend(new_column_names)
adb_data_df.columns = titles
non_adb_df.columns = titles

print(non_adb_df)

In [ ]:
column_titles = ['start_unix', 'end_unix', 'mean_nni', 'sdnn', 'sdsd', 'nni_50', 'pnni_50', 'nni_20', 'pnni_20', 'rmssd', 'median_nni', 'range_nni', 'cvsd', 'cvnni', 'mean_hr', 'max_hr', 'min_hr', 'std_hr', 'lf', 'hf', 'lf_hf_ratio', 'lfnu', 'hfnu', 'total_power', 'vlf']

df_concatenate_adb = pd.DataFrame()

for title in column_titles:
    df_column_group = pd.DataFrame()

    for i in range(0, 7500, 25):
        column = adb_data_df.iloc[:, i + column_titles.index(title)] 
        df_column_group = pd.concat([df_column_group, column], axis=1) 

    df_concatenate_adb = pd.concat([df_concatenate_adb, df_column_group], axis=1)
    adb_df_final = pd.concat([df_concatenate_adb, sleep_data_adb], axis=1)
print(adb_df_final)

In [ ]:
adb_df_final.to_csv('time_series_adb.csv', index=False)

In [ ]:
column_titles = ['start_unix', 'end_unix', 'mean_nni', 'sdnn', 'sdsd', 'nni_50', 'pnni_50', 'nni_20', 'pnni_20', 'rmssd', 'median_nni', 'range_nni', 'cvsd', 'cvnni', 'mean_hr', 'max_hr', 'min_hr', 'std_hr', 'lf', 'hf', 'lf_hf_ratio', 'lfnu', 'hfnu', 'total_power', 'vlf']

df_concatenate_non_adb = pd.DataFrame()

for title in column_titles:
    df_column_group = pd.DataFrame()

    for i in range(0, 7500, 25):
        column = non_adb_df.iloc[:, i + column_titles.index(title)] 
        df_column_group = pd.concat([df_column_group, column], axis=1) 

    df_concatenate_non_adb = pd.concat([df_concatenate_non_adb, df_column_group], axis=1)

print(df_concatenate_non_adb)

In [ ]:
non_adb_df_final = pd.concat([df_concatenate_non_adb, sleep_data_non_adb], axis=1)
print(non_adb_df_final)

In [ ]:
non_adb_df_final.to_csv('time_series_non_adb.csv', index=False)
# df_concatenated.to_csv('time_series_non_adb.csv', index=False)